# CS201L: Artificial Intelligence Laboratory
## Lab 13: Hierarchical and DBSCAN Clustering
**Indian Institute of Technology Dharwad**

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import AgglomerativeClustering, DBSCAN
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.metrics import normalized_mutual_info_score, silhouette_score
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

## Helper Function: Purity Score

In [ ]:
def purity_score(true_labels, cluster_labels):
    """
    Compute purity score for clustering.
    Purity = (1/N) * sum_k( max_j( |cluster_k ∩ class_j| ) )
    """
    N = len(true_labels)
    total_correct = 0
    for cluster_id in np.unique(cluster_labels):
        if cluster_id == -1:  # Skip noise points in DBSCAN
            continue
        cluster_mask = cluster_labels == cluster_id
        true_in_cluster = true_labels[cluster_mask]
        if len(true_in_cluster) > 0:
            most_common_count = Counter(true_in_cluster).most_common(1)[0][1]
            total_correct += most_common_count
    return total_correct / N

## 1. Data Preprocessing and Dimensionality Reduction

In [ ]:
# (a) Load the Iris dataset
iris_df = pd.read_csv('Iris.csv')
print('Shape:', iris_df.shape)
print('\nFirst few rows:')
print(iris_df.head())
print('\nColumn names:')
print(iris_df.columns.tolist())
print('\nSpecies distribution:')
print(iris_df.iloc[:, -1].value_counts())

In [ ]:
# (b) Apply PCA to reduce to 2D
# Extract features (first 4 columns) and labels (last column)
features = iris_df.iloc[:, :-1].values
true_labels = iris_df.iloc[:, -1].values

print('Original features shape:', features.shape)

# Apply PCA
pca = PCA(n_components=2)
features_2d = pca.fit_transform(features)

print('Reduced features shape:', features_2d.shape)
print('\nExplained variance ratio:', pca.explained_variance_ratio_)
print('Total variance explained:', sum(pca.explained_variance_ratio_))

In [ ]:
# (c) Save 2D dataset and visualize
iris_2d_df = pd.DataFrame({
    'PC1': features_2d[:, 0],
    'PC2': features_2d[:, 1],
    'Species': true_labels
})

iris_2d_df.to_csv('iris_dataset_2D.csv', index=False)
print('2D dataset saved as iris_dataset_2D.csv')

# Visualize
species = np.unique(true_labels)
colors = ['red', 'blue', 'green']

plt.figure(figsize=(10, 7))
for sp, color in zip(species, colors):
    mask = true_labels == sp
    plt.scatter(features_2d[mask, 0], features_2d[mask, 1],
                label=sp, color=color, alpha=0.6, s=50, edgecolors='black')

plt.title('Iris Dataset - 2D PCA Projection', fontsize=14, fontweight='bold')
plt.xlabel('First Principal Component (PC1)', fontsize=12)
plt.ylabel('Second Principal Component (PC2)', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('iris_2d_pca.png', dpi=150)
plt.show()

## 2. Hierarchical Clustering and Comparison

In [ ]:
# (a) Load the iris_dataset_2D
iris_2d = pd.read_csv('iris_dataset_2D.csv')
X_2d = iris_2d[['PC1', 'PC2']].values
y_true = iris_2d['Species'].values

print('Loaded 2D dataset shape:', X_2d.shape)

### 2.1 Ward Linkage

In [ ]:
# (b) Agglomerative Clustering with Ward linkage
agg_ward = AgglomerativeClustering(n_clusters=3, linkage='ward')
labels_ward = agg_ward.fit_predict(X_2d)

# Visualize
plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=labels_ward, 
                      cmap='viridis', alpha=0.6, s=50, edgecolors='black')
plt.colorbar(scatter, label='Cluster')
plt.title('Hierarchical Clustering - Ward Linkage (K=3)', fontsize=14, fontweight='bold')
plt.xlabel('PC1', fontsize=12)
plt.ylabel('PC2', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('hierarchical_ward.png', dpi=150)
plt.show()

In [ ]:
# (c) Compute metrics for Ward linkage
purity_ward = purity_score(y_true, labels_ward)
nmi_ward = normalized_mutual_info_score(y_true, labels_ward)
sil_ward = silhouette_score(X_2d, labels_ward)

print('=== Ward Linkage Metrics ===')
print(f'Purity Score: {purity_ward:.4f}')
print(f'NMI Score: {nmi_ward:.4f}')
print(f'Silhouette Score: {sil_ward:.4f}')

### 2.2 Complete Linkage

In [ ]:
# (d) Agglomerative Clustering with Complete linkage
agg_complete = AgglomerativeClustering(n_clusters=3, linkage='complete')
labels_complete = agg_complete.fit_predict(X_2d)

# Visualize
plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=labels_complete, 
                      cmap='plasma', alpha=0.6, s=50, edgecolors='black')
plt.colorbar(scatter, label='Cluster')
plt.title('Hierarchical Clustering - Complete Linkage (K=3)', fontsize=14, fontweight='bold')
plt.xlabel('PC1', fontsize=12)
plt.ylabel('PC2', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('hierarchical_complete.png', dpi=150)
plt.show()

In [ ]:
# (e) Compute metrics for Complete linkage
purity_complete = purity_score(y_true, labels_complete)
nmi_complete = normalized_mutual_info_score(y_true, labels_complete)
sil_complete = silhouette_score(X_2d, labels_complete)

print('=== Complete Linkage Metrics ===')
print(f'Purity Score: {purity_complete:.4f}')
print(f'NMI Score: {nmi_complete:.4f}')
print(f'Silhouette Score: {sil_complete:.4f}')

### 2.3 Average Linkage

In [ ]:
# (f) Agglomerative Clustering with Average linkage
agg_average = AgglomerativeClustering(n_clusters=3, linkage='average')
labels_average = agg_average.fit_predict(X_2d)

# Visualize
plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=labels_average, 
                      cmap='coolwarm', alpha=0.6, s=50, edgecolors='black')
plt.colorbar(scatter, label='Cluster')
plt.title('Hierarchical Clustering - Average Linkage (K=3)', fontsize=14, fontweight='bold')
plt.xlabel('PC1', fontsize=12)
plt.ylabel('PC2', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('hierarchical_average.png', dpi=150)
plt.show()

In [ ]:
# (g) Compute metrics for Average linkage
purity_average = purity_score(y_true, labels_average)
nmi_average = normalized_mutual_info_score(y_true, labels_average)
sil_average = silhouette_score(X_2d, labels_average)

print('=== Average Linkage Metrics ===')
print(f'Purity Score: {purity_average:.4f}')
print(f'NMI Score: {nmi_average:.4f}')
print(f'Silhouette Score: {sil_average:.4f}')

### 2.4 Single Linkage

In [ ]:
# (h) Agglomerative Clustering with Single linkage
agg_single = AgglomerativeClustering(n_clusters=3, linkage='single')
labels_single = agg_single.fit_predict(X_2d)

# Visualize
plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=labels_single, 
                      cmap='Spectral', alpha=0.6, s=50, edgecolors='black')
plt.colorbar(scatter, label='Cluster')
plt.title('Hierarchical Clustering - Single Linkage (K=3)', fontsize=14, fontweight='bold')
plt.xlabel('PC1', fontsize=12)
plt.ylabel('PC2', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('hierarchical_single.png', dpi=150)
plt.show()

In [ ]:
# (i) Compute metrics for Single linkage
purity_single = purity_score(y_true, labels_single)
nmi_single = normalized_mutual_info_score(y_true, labels_single)
sil_single = silhouette_score(X_2d, labels_single)

print('=== Single Linkage Metrics ===')
print(f'Purity Score: {purity_single:.4f}')
print(f'NMI Score: {nmi_single:.4f}')
print(f'Silhouette Score: {sil_single:.4f}')

### 2.5 Comparison of All Linkage Methods

In [ ]:
# Create comparison table
comparison_df = pd.DataFrame({
    'Linkage': ['Ward', 'Complete', 'Average', 'Single'],
    'Purity': [purity_ward, purity_complete, purity_average, purity_single],
    'NMI': [nmi_ward, nmi_complete, nmi_average, nmi_single],
    'Silhouette': [sil_ward, sil_complete, sil_average, sil_single]
})

print('\n=== Hierarchical Clustering Comparison (K=3) ===')
print(comparison_df.to_string(index=False))

# Find best linkage based on purity
best_idx = comparison_df['Purity'].idxmax()
best_linkage = comparison_df.loc[best_idx, 'Linkage']
print(f'\nBest linkage method based on Purity: {best_linkage}')

### 2.6 Dendrogram Analysis

In [ ]:
# (j) Plot dendrogram using the best linkage method
best_linkage_lower = best_linkage.lower()
linked = linkage(X_2d, method=best_linkage_lower)

plt.figure(figsize=(14, 7))
dendrogram(linked, 
           truncate_mode='lastp',
           p=30,
           leaf_rotation=90,
           leaf_font_size=10,
           show_contracted=True)

plt.axhline(y=6, color='r', linestyle='--', linewidth=2, label='Threshold = 6')
plt.title(f'Hierarchical Clustering Dendrogram ({best_linkage} Linkage)', 
          fontsize=14, fontweight='bold')
plt.xlabel('Sample Index or (Cluster Size)', fontsize=12)
plt.ylabel('Distance', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('dendrogram.png', dpi=150)
plt.show()

# Determine optimal number of clusters at threshold=6
clusters_at_threshold = fcluster(linked, t=6, criterion='distance')
optimal_k = len(np.unique(clusters_at_threshold))
print(f'\nOptimal number of clusters at threshold=6: {optimal_k}')

In [ ]:
# (k) Apply clustering with optimal K
agg_optimal = AgglomerativeClustering(n_clusters=optimal_k, linkage=best_linkage_lower)
labels_optimal = agg_optimal.fit_predict(X_2d)

print(f'Applied Agglomerative Clustering with K={optimal_k} using {best_linkage} linkage')

In [ ]:
# (l) Visualize optimal clustering
plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=labels_optimal, 
                      cmap='tab10', alpha=0.6, s=50, edgecolors='black')
plt.colorbar(scatter, label='Cluster')
plt.title(f'Hierarchical Clustering - {best_linkage} Linkage (K={optimal_k})', 
          fontsize=14, fontweight='bold')
plt.xlabel('PC1', fontsize=12)
plt.ylabel('PC2', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('hierarchical_optimal.png', dpi=150)
plt.show()

# Compute metrics
purity_optimal = purity_score(y_true, labels_optimal)
nmi_optimal = normalized_mutual_info_score(y_true, labels_optimal)
sil_optimal = silhouette_score(X_2d, labels_optimal)

print(f'\n=== Optimal Hierarchical Clustering Metrics (K={optimal_k}) ===')
print(f'Purity Score: {purity_optimal:.4f}')
print(f'NMI Score: {nmi_optimal:.4f}')
print(f'Silhouette Score: {sil_optimal:.4f}')

In [ ]:
# (n) Comparison with K-Means and K-Medoids from Lab 12
# Note: These values should be taken from your Lab 12 results for K=3
# Replace these with actual values from Lab 12
kmeans_purity_k3 = 0.8933  # Example value - replace with actual
kmedoid_purity_k3 = 0.8867  # Example value - replace with actual

print('\n=== Comparison: Hierarchical vs K-Means vs K-Medoids (K=3) ===')
print(f"{'Method':<25} {'Purity Score':>15}")
print('-' * 42)
print(f"{'Hierarchical (Ward)':<25} {purity_ward:>15.4f}")
print(f"{'Hierarchical (Complete)':<25} {purity_complete:>15.4f}")
print(f"{'Hierarchical (Average)':<25} {purity_average:>15.4f}")
print(f"{'Hierarchical (Single)':<25} {purity_single:>15.4f}")
print(f"{'K-Means (from Lab 12)':<25} {kmeans_purity_k3:>15.4f}")
print(f"{'K-Medoids (from Lab 12)':<25} {kmedoid_purity_k3:>15.4f}")

print('\nNote: K-Means and K-Medoids values are examples. Replace with actual values from Lab 12.')

## 3. DBSCAN Clustering

In [ ]:
# (a) Load the iris_dataset_2D again
iris_2d = pd.read_csv('iris_dataset_2D.csv')
X_2d = iris_2d[['PC1', 'PC2']].values
y_true = iris_2d['Species'].values

print('Loaded 2D dataset for DBSCAN')

### 3.1 DBSCAN with Default Parameters

In [ ]:
# (b) DBSCAN with default parameters
dbscan_default = DBSCAN(eps=0.5, min_samples=5)
labels_dbscan_default = dbscan_default.fit_predict(X_2d)

n_clusters_default = len(set(labels_dbscan_default)) - (1 if -1 in labels_dbscan_default else 0)
n_noise_default = list(labels_dbscan_default).count(-1)

print(f'\n=== DBSCAN Default (eps=0.5, min_samples=5) ===')
print(f'Number of clusters: {n_clusters_default}')
print(f'Number of noise points: {n_noise_default}')

In [ ]:
# (i) Visualize default DBSCAN
plt.figure(figsize=(10, 7))
unique_labels = set(labels_dbscan_default)
colors_dbscan = plt.cm.Spectral(np.linspace(0, 1, len(unique_labels)))

for k, col in zip(unique_labels, colors_dbscan):
    if k == -1:
        col = [0, 0, 0, 1]  # Black for noise
    
    class_member_mask = (labels_dbscan_default == k)
    xy = X_2d[class_member_mask]
    
    if k == -1:
        plt.scatter(xy[:, 0], xy[:, 1], c=[col], marker='x', s=50, label='Noise')
    else:
        plt.scatter(xy[:, 0], xy[:, 1], c=[col], marker='o', s=50, 
                   edgecolors='black', alpha=0.6, label=f'Cluster {k}')

plt.title('DBSCAN Clustering - Default (eps=0.5, min_samples=5)', 
          fontsize=14, fontweight='bold')
plt.xlabel('PC1', fontsize=12)
plt.ylabel('PC2', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('dbscan_default.png', dpi=150)
plt.show()

In [ ]:
# (ii) Compute metrics for default DBSCAN
if n_clusters_default > 1:
    purity_dbscan_default = purity_score(y_true, labels_dbscan_default)
    nmi_dbscan_default = normalized_mutual_info_score(y_true, labels_dbscan_default)
    # Only compute silhouette if there are at least 2 clusters
    if len(set(labels_dbscan_default)) > 1 and n_clusters_default > 1:
        # Exclude noise points for silhouette score
        mask = labels_dbscan_default != -1
        if sum(mask) > 0 and len(set(labels_dbscan_default[mask])) > 1:
            sil_dbscan_default = silhouette_score(X_2d[mask], labels_dbscan_default[mask])
        else:
            sil_dbscan_default = None
    else:
        sil_dbscan_default = None
else:
    purity_dbscan_default = None
    nmi_dbscan_default = None
    sil_dbscan_default = None

print('\n=== DBSCAN Default Metrics ===')
print(f'Purity Score: {purity_dbscan_default:.4f}' if purity_dbscan_default else 'Purity Score: N/A')
print(f'NMI Score: {nmi_dbscan_default:.4f}' if nmi_dbscan_default else 'NMI Score: N/A')
print(f'Silhouette Score: {sil_dbscan_default:.4f}' if sil_dbscan_default else 'Silhouette Score: N/A')

### 3.2 DBSCAN with Different Parameters

In [ ]:
# (c) DBSCAN with different parameters
eps_values = [1, 5]
min_samples_values = [4, 5, 7, 10]

dbscan_results = []

for eps in eps_values:
    for min_samp in min_samples_values:
        dbscan = DBSCAN(eps=eps, min_samples=min_samp)
        labels = dbscan.fit_predict(X_2d)
        
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = list(labels).count(-1)
        
        # Compute metrics
        if n_clusters > 1:
            purity = purity_score(y_true, labels)
            nmi = normalized_mutual_info_score(y_true, labels)
            mask = labels != -1
            if sum(mask) > 0 and len(set(labels[mask])) > 1:
                sil = silhouette_score(X_2d[mask], labels[mask])
            else:
                sil = None
        else:
            purity = None
            nmi = None
            sil = None
        
        dbscan_results.append({
            'eps': eps,
            'min_samples': min_samp,
            'n_clusters': n_clusters,
            'n_noise': n_noise,
            'labels': labels,
            'purity': purity,
            'nmi': nmi,
            'silhouette': sil
        })
        
        print(f'eps={eps}, min_samples={min_samp}: Clusters={n_clusters}, Noise={n_noise}')

In [ ]:
# (i) Visualize all DBSCAN configurations
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()

for idx, result in enumerate(dbscan_results):
    ax = axes[idx]
    labels = result['labels']
    eps = result['eps']
    min_samp = result['min_samples']
    
    unique_labels = set(labels)
    colors_dbscan = plt.cm.Spectral(np.linspace(0, 1, len(unique_labels)))
    
    for k, col in zip(unique_labels, colors_dbscan):
        if k == -1:
            col = [0, 0, 0, 1]
        
        class_member_mask = (labels == k)
        xy = X_2d[class_member_mask]
        
        if k == -1:
            ax.scatter(xy[:, 0], xy[:, 1], c=[col], marker='x', s=30)
        else:
            ax.scatter(xy[:, 0], xy[:, 1], c=[col], marker='o', s=30, 
                      edgecolors='black', alpha=0.6)
    
    ax.set_title(f'eps={eps}, min_samples={min_samp}\nClusters={result["n_clusters"]}', 
                fontsize=10)
    ax.set_xlabel('PC1', fontsize=9)
    ax.set_ylabel('PC2', fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('DBSCAN Clustering - Different Parameters', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('dbscan_all_params.png', dpi=150)
plt.show()

In [ ]:
# (ii) Create comparison table
dbscan_comparison = pd.DataFrame([
    {
        'eps': r['eps'],
        'min_samples': r['min_samples'],
        'Clusters': r['n_clusters'],
        'Noise Points': r['n_noise'],
        'Purity': f"{r['purity']:.4f}" if r['purity'] is not None else 'N/A',
        'NMI': f"{r['nmi']:.4f}" if r['nmi'] is not None else 'N/A',
        'Silhouette': f"{r['silhouette']:.4f}" if r['silhouette'] is not None else 'N/A'
    }
    for r in dbscan_results
])

print('\n=== DBSCAN Performance Comparison ===')
print(dbscan_comparison.to_string(index=False))

In [ ]:
# (iii) Find optimal DBSCAN configuration and display frequent species
# Find configuration with 3 clusters (matching true number of species)
optimal_configs = [r for r in dbscan_results if r['n_clusters'] == 3]

if optimal_configs:
    # Select based on highest purity among 3-cluster configs
    optimal_config = max([r for r in optimal_configs if r['purity'] is not None], 
                         key=lambda x: x['purity'])
    
    print(f"\n=== Optimal DBSCAN Configuration (3 clusters) ===")
    print(f"eps={optimal_config['eps']}, min_samples={optimal_config['min_samples']}")
    print(f"Purity: {optimal_config['purity']:.4f}")
    print(f"NMI: {optimal_config['nmi']:.4f}")
    if optimal_config['silhouette']:
        print(f"Silhouette: {optimal_config['silhouette']:.4f}")
    
    # Find frequent species in each cluster
    optimal_labels = optimal_config['labels']
    print("\n=== Frequent Species in Each Cluster ===")
    for cluster_id in sorted(set(optimal_labels)):
        if cluster_id == -1:
            cluster_name = "Noise"
        else:
            cluster_name = f"Cluster {cluster_id}"
        
        mask = optimal_labels == cluster_id
        species_in_cluster = y_true[mask]
        
        if len(species_in_cluster) > 0:
            species_count = Counter(species_in_cluster)
            most_common = species_count.most_common(1)[0]
            print(f"{cluster_name}: {most_common[0]} ({most_common[1]} samples)")
            print(f"  Distribution: {dict(species_count)}")
else:
    print("\nNo configuration found with exactly 3 clusters.")
    print("Showing configuration with highest purity:")
    best_config = max([r for r in dbscan_results if r['purity'] is not None], 
                      key=lambda x: x['purity'])
    print(f"eps={best_config['eps']}, min_samples={best_config['min_samples']}")
    print(f"Clusters: {best_config['n_clusters']}, Purity: {best_config['purity']:.4f}")

## 4. Final Comparison: All Methods

In [ ]:
# Compare all clustering methods
print('\n' + '='*60)
print('FINAL COMPARISON: ALL CLUSTERING METHODS')
print('='*60)

print(f"\n{'Method':<30} {'Purity':>10} {'NMI':>10} {'Silhouette':>12}")
print('-' * 65)

# Hierarchical methods
print(f"{'Hierarchical - Ward':<30} {purity_ward:>10.4f} {nmi_ward:>10.4f} {sil_ward:>12.4f}")
print(f"{'Hierarchical - Complete':<30} {purity_complete:>10.4f} {nmi_complete:>10.4f} {sil_complete:>12.4f}")
print(f"{'Hierarchical - Average':<30} {purity_average:>10.4f} {nmi_average:>10.4f} {sil_average:>12.4f}")
print(f"{'Hierarchical - Single':<30} {purity_single:>10.4f} {nmi_single:>10.4f} {sil_single:>12.4f}")

# DBSCAN
if purity_dbscan_default:
    sil_str = f"{sil_dbscan_default:.4f}" if sil_dbscan_default else "N/A"
    print(f"{'DBSCAN (default)':<30} {purity_dbscan_default:>10.4f} {nmi_dbscan_default:>10.4f} {sil_str:>12}")

if optimal_configs:
    sil_str = f"{optimal_config['silhouette']:.4f}" if optimal_config['silhouette'] else "N/A"
    print(f"{'DBSCAN (optimal)':<30} {optimal_config['purity']:>10.4f} {optimal_config['nmi']:>10.4f} {sil_str:>12}")

# Note about K-Means and K-Medoids
print(f"\n{'K-Means (from Lab 12)':<30} {kmeans_purity_k3:>10.4f} {'--':>10} {'--':>12}")
print(f"{'K-Medoids (from Lab 12)':<30} {kmedoid_purity_k3:>10.4f} {'--':>10} {'--':>12}")
print("\n(Replace K-Means and K-Medoids values with actual results from Lab 12)")

print('\n' + '='*60)